### Q1

In [22]:
%%writefile task_sum.cu
#include <stdio.h>

__global__ void taskKernel(int n, int *results) {
    int tid = threadIdx.x;

    // Thread 0: Iterative approach
    if (tid == 0) {
        int sum = 0;
        for (int i = 1; i <= n; i++) sum += i;
        results[0] = sum;
    }
    // Thread 1: Direct formula approach
    else if (tid == 1) {
        results[1] = (n * (n + 1)) / 2;
    }
}

int main() {
    int n = 1024;
    int h_results[2];
    int *d_results;

    cudaMalloc(&d_results, 2 * sizeof(int));
    taskKernel<<<1, 2>>>(n, d_results);
    cudaMemcpy(h_results, d_results, 2 * sizeof(int), cudaMemcpyDeviceToHost);

    printf("Iterative Sum: %d\n", h_results[0]);
    printf("Formula Sum: %d\n", h_results[1]);

    cudaFree(d_results);
    return 0;
}

Overwriting task_sum.cu


In [23]:
!nvcc task_sum.cu -o task_sum && ./task_sum

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Iterative Sum: 524800
Formula Sum: 524800


### Q2

In [24]:
%%writefile merge_sort.cu
#include <stdio.h>
#include <cuda_runtime.h>

__device__ void gpu_merge(int* data, int* temp, int left, int mid, int right) {
    int i = left, j = mid, k = left;
    while (i < mid && j < right) {
        if (data[i] <= data[j]) temp[k++] = data[i++];
        else temp[k++] = data[j++];
    }
    while (i < mid) temp[k++] = data[i++];
    while (j < right) temp[k++] = data[j++];
    for (int l = left; l < right; l++) data[l] = temp[l];
}

__global__ void mergeSortKernel(int* data, int* temp, int size, int width) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int left = idx * width * 2;
    if (left < size) {
        int mid = left + width;
        int right = (left + 2 * width < size) ? (left + 2 * width) : size;
        if (mid < size) gpu_merge(data, temp, left, mid, right);
    }
}

int main() {
    const int n = 1000;
    int h_data[n];
    for(int i=0; i<n; i++) h_data[i] = n - i;

    int *d_data, *d_temp;
    cudaMalloc(&d_data, n * sizeof(int));
    cudaMalloc(&d_temp, n * sizeof(int));
    cudaMemcpy(d_data, h_data, n * sizeof(int), cudaMemcpyHostToDevice);

    for (int width = 1; width < n; width *= 2) {
        int numBlocks = (n / (2 * width)) + 1;
        mergeSortKernel<<<numBlocks, 1>>>(d_data, d_temp, n, width);
        cudaDeviceSynchronize();
    }

    cudaMemcpy(h_data, d_data, n * sizeof(int), cudaMemcpyDeviceToHost);
    printf("First 5 sorted: %d %d %d %d %d\n", h_data[0], h_data[1], h_data[2], h_data[3], h_data[4]);

    cudaFree(d_data); cudaFree(d_temp);
    return 0;
}

Overwriting merge_sort.cu


In [25]:
!nvcc merge_sort.cu -o merge_sort && ./merge_sort

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
First 5 sorted: 1 2 3 4 5


### Q3

In [26]:
%%writefile vector_add.cu
#include <stdio.h>

#define N 100000
__device__ float dev_a[N];
__device__ float dev_b[N];
__device__ float dev_c[N];

__global__ void vectorAdd() {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) dev_c[i] = dev_a[i] + dev_b[i];
}

int main() {
    float h_a[N], h_b[N];
    for(int i=0; i<N; i++) { h_a[i] = 1.0f; h_b[i] = 2.0f; }

    cudaMemcpyToSymbol(dev_a, h_a, N * sizeof(float));
    cudaMemcpyToSymbol(dev_b, h_b, N * sizeof(float));

    cudaEvent_t start, stop;
    cudaEventCreate(&start); cudaEventCreate(&stop);
    cudaEventRecord(start);

    vectorAdd<<<(N+255)/256, 256>>>();

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    cudaDeviceProp prop;
    cudaGetDeviceProperties(&prop, 0);

    double theoreticalBW = (double)prop.memoryClockRate * 1000.0 * (prop.memoryBusWidth / 8.0) * 2.0 / 1e9;
    double bytesProcessed = 3.0 * N * sizeof(float);
    double measuredBW = (bytesProcessed / (milliseconds / 1000.0)) / 1e9;

    printf("Time: %f ms\n", milliseconds);
    printf("Theoretical Bandwidth: %f GB/s\n", theoreticalBW);
    printf("Measured Bandwidth: %f GB/s\n", measuredBW);

    return 0;
}

Overwriting vector_add.cu


In [27]:
!nvcc -O3 vector_add.cu -o vector_add && ./vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Time: 0.051264 ms
Theoretical Bandwidth: 320.064000 GB/s
Measured Bandwidth: 23.408240 GB/s
